In [ ]:
#!/usr/bin/env python3

from pathlib import Path
import re
import random
import shutil
import numpy as np
import soundfile as sf
from scipy.io import wavfile
import os
import stat
import time

# =========================================================
# SETTINGS
# =========================================================
ROOT = Path(".").resolve()

KEYWORDS_DIR = ROOT / "keywords"
UNKNOWN_DIR = ROOT / "unknown"

KEYWORDS_SPLICED_DIR = KEYWORDS_DIR / "spliced"
RANDOM_DIR = UNKNOWN_DIR / "random"
RANDOM_SPLICED_DIR = RANDOM_DIR / "spliced"
PREPARED_WORDS_DIR = ROOT / "prepared_words"

TARGET_LABELS = ["får", "ged", "hest", "laks", "ulv"]
UNKNOWN_LABELS = ["random", "stilhed"]

SAMPLE_RATE = 16000
SAMPLE_TIME = 1.0
BIT_DEPTH = "PCM_16"

# ---------- splicing params for keywords ----------
WINDOW_S = 1.00
PRE_S = 0.20
MIN_GAP_S = 0.45
FRAME_MS = 20
HOP_MS = 10
THRESH_MULT = 6.0
MAX_SLICES_PER_FILE = 10

# Special tuning for difficult files
SPECIAL_FILE_PARAMS = {
    "laks_C_0.75m_stille.001.wav": {
        "window_s": 1.0,
        "pre_s": 0.22,
        "min_gap_s": 0.55,
        "frame_ms": 20,
        "hop_ms": 10,
        "thresh_mult": 4.5,
        "max_slices": 10,
    },
    "laks_C_1.5m_stille.001.wav": {
        "window_s": 1.0,
        "pre_s": 0.25,
        "min_gap_s": 0.60,
        "frame_ms": 20,
        "hop_ms": 10,
        "thresh_mult": 4.0,
        "max_slices": 10,
    },
}

FNAME_RE = re.compile(
    r"^(?P<label>[^_]+)_(?P<speaker>[A-Za-z])_(?P<dist>[0-9.]+m)_(?P<env>[^.]+)\.(?P<take>\d+)\.wav$",
    re.IGNORECASE
)

SPLICED_FNAME_RE = re.compile(
    r"^(?P<label>.+?)_(?P<speaker>[A-Za-z])_(?P<dist>[0-9.]+m)_(?P<env>[^_]+)_(?P<take>\d+)_s(?P<slice>\d+)\.wav$",
    re.IGNORECASE
)



# =========================================================
# HELPERS
# =========================================================

def _handle_remove_readonly(func, path, exc):
    """
    Bruges af shutil.rmtree hvis Windows nægter adgang pga. read-only/lås.
    """
    try:
        os.chmod(path, stat.S_IWRITE)
        func(path)
    except Exception as e:
        print(f"[reset] Could not delete {path}: {e}")

def safe_rmtree(path: Path, retries: int = 5, delay: float = 0.5):
    """
    Forsøger at slette en mappe flere gange. Hjælper mod midlertidige fil-låse.
    """
    if not path.exists():
        return

    for attempt in range(1, retries + 1):
        try:
            shutil.rmtree(path, onerror=_handle_remove_readonly)
            return
        except PermissionError as e:
            print(f"[reset] PermissionError deleting {path} (attempt {attempt}/{retries}): {e}")
            time.sleep(delay)

    raise PermissionError(f"Could not delete folder after {retries} attempts: {path}")

def reset_output_dirs():
    paths_to_reset = [
        KEYWORDS_SPLICED_DIR,
        RANDOM_SPLICED_DIR,
        PREPARED_WORDS_DIR,
    ]

    for unknown_label in UNKNOWN_LABELS:
        if unknown_label != "random":
            paths_to_reset.append(UNKNOWN_DIR / unknown_label / "spliced")

    for path in paths_to_reset:
        if path.exists():
            print(f"[reset] Deleting: {path}")
            safe_rmtree(path)

def sanitize(s: str) -> str:
    s = s.strip()
    return re.sub(r'[\\/:*?"<>|]+', "_", s)

def to_mono(x: np.ndarray) -> np.ndarray:
    if x.ndim == 1:
        return x
    return x.mean(axis=1)

def moving_rms(x: np.ndarray, frame: int, hop: int) -> np.ndarray:
    if len(x) < frame:
        return np.array([], dtype=np.float32)
    n = 1 + (len(x) - frame) // hop
    rms = np.empty(n, dtype=np.float32)
    for i in range(n):
        seg = x[i * hop : i * hop + frame]
        rms[i] = np.sqrt(np.mean(seg * seg) + 1e-12)
    return rms

def detect_event_starts(rms: np.ndarray, thr: float, hop: int) -> np.ndarray:
    if rms.size == 0:
        return np.array([], dtype=np.int64)
    above = rms > thr
    starts = np.where(np.logical_and(above, np.concatenate([[False], ~above[:-1]])))[0]
    return (starts * hop).astype(np.int64)

def parse_base_name(wav_path: Path) -> str:
    m = FNAME_RE.match(wav_path.name)
    if m:
        label = sanitize(m.group("label"))
        speaker = sanitize(m.group("speaker"))
        dist = sanitize(m.group("dist"))
        env = sanitize(m.group("env"))
        take = int(m.group("take"))
        return f"{label}_{speaker}_{dist}_{env}_{take:03d}"
    return sanitize(wav_path.stem)

def ensure_clean_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def load_audio_fixed(path: Path, sample_rate: int, sample_time: float):
    x, fs = sf.read(str(path), always_2d=False)
    x = np.asarray(x, dtype=np.float32)
    x = to_mono(x)

    if fs != sample_rate:
        old_idx = np.linspace(0, len(x) - 1, num=len(x), dtype=np.float32)
        new_len = int(round(len(x) * sample_rate / fs))
        new_idx = np.linspace(0, len(x) - 1, num=new_len, dtype=np.float32)
        x = np.interp(new_idx, old_idx, x).astype(np.float32)

    target_len = int(sample_rate * sample_time)
    if len(x) < target_len:
        x = np.pad(x, (0, target_len - len(x)))
    else:
        x = x[:target_len]

    return x

def get_random_bg_snippet(bg_path: Path, sample_rate: int, sample_time: float):
    bg, fs = sf.read(str(bg_path), always_2d=False)
    bg = np.asarray(bg, dtype=np.float32)
    bg = to_mono(bg)

    if fs != sample_rate:
        old_idx = np.linspace(0, len(bg) - 1, num=len(bg), dtype=np.float32)
        new_len = int(round(len(bg) * sample_rate / fs))
        new_idx = np.linspace(0, len(bg) - 1, num=new_len, dtype=np.float32)
        bg = np.interp(new_idx, old_idx, bg).astype(np.float32)

    target_len = int(sample_rate * sample_time)
    if len(bg) < target_len:
        raise ValueError(f"Background file too short: {bg_path}")

    max_start = len(bg) - target_len
    start = random.randint(0, max_start)
    return bg[start:start + target_len]

def normalize_peak(x: np.ndarray, eps: float = 1e-9):
    peak = np.max(np.abs(x))
    if peak < eps:
        return x
    return x / peak

#def mix_audio(word_waveform, bg_waveform, word_vol=1.0, bg_vol=0.35):
#    word_waveform = normalize_peak(word_waveform)
#    bg_waveform = normalize_peak(bg_waveform)
#    mixed = word_vol * word_waveform + bg_vol * bg_waveform
#    mixed = np.clip(mixed, -1.0, 1.0)
#    return mixed.astype(np.float32)

def mix_audio(word_waveform, bg_waveform, word_vol=1.0, bg_vol=0.35):
    mixed = word_vol * word_waveform + bg_vol * bg_waveform

    peak = np.max(np.abs(mixed))
    if peak > 1.0:
        mixed = mixed / peak

    return mixed.astype(np.float32)

def replace_noise_label(filename: str, new_noise_label: str):
    if "_stille_" in filename:
        return filename.replace("_stille_", f"_{new_noise_label}_", 1)
    stem = Path(filename).stem
    suffix = Path(filename).suffix
    return f"{stem}_{new_noise_label}{suffix}"

# =========================================================
# STEP 1: SPLICE KEYWORDS
# =========================================================
def slice_keyword_file(wav_path: Path, out_dir: Path, cfg: dict) -> int:
    fs, data = wavfile.read(wav_path)
    x = to_mono(data).astype(np.float32)

    frame = int(fs * (cfg["frame_ms"] / 1000.0))
    hop = int(fs * (cfg["hop_ms"] / 1000.0))

    rms = moving_rms(x, frame, hop)
    if rms.size == 0:
        return 0

    med = float(np.median(rms))
    thr = max(med * cfg["thresh_mult"], 1e-6)

    starts = detect_event_starts(rms, thr, hop)

    min_gap = int(fs * cfg["min_gap_s"])
    filtered = []
    last = -10**18
    for s in starts:
        if s - last >= min_gap:
            filtered.append(s)
            last = s
        if len(filtered) >= cfg["max_slices"]:
            break

    if not filtered:
        return 0

    base = parse_base_name(wav_path)
    out_dir.mkdir(parents=True, exist_ok=True)

    win = int(fs * cfg["window_s"])
    pre = int(fs * cfg["pre_s"])

    written = 0
    for i, s in enumerate(filtered, start=1):
        start = max(int(s - pre), 0)
        end = start + win
        if end > len(x):
            continue

        clip = x[start:end]
        clip = np.clip(clip, -32768, 32767).astype(np.int16)
        out_path = out_dir / f"{base}_s{i:02d}.wav"
        wavfile.write(out_path, fs, clip)
        written += 1

    return written

def splice_keywords():
    ensure_clean_dir(KEYWORDS_SPLICED_DIR)

    wavs = sorted([p for p in KEYWORDS_DIR.glob("*.wav") if p.is_file()])
    print(f"[keywords] Found {len(wavs)} raw files")

    total = 0
    default_cfg = {
        "window_s": WINDOW_S,
        "pre_s": PRE_S,
        "min_gap_s": MIN_GAP_S,
        "frame_ms": FRAME_MS,
        "hop_ms": HOP_MS,
        "thresh_mult": THRESH_MULT,
        "max_slices": MAX_SLICES_PER_FILE,
    }

    for wav in wavs:
        cfg = SPECIAL_FILE_PARAMS.get(wav.name, default_cfg)
        n = slice_keyword_file(wav, KEYWORDS_SPLICED_DIR, cfg)
        print(f"[keywords] {wav.name}: {n} slices")
        total += n

    print(f"[keywords] Total slices: {total}")

# =========================================================
# STEP 2: SPLICE RANDOM WORDS
# =========================================================
def splice_random_words():
    ensure_clean_dir(RANDOM_SPLICED_DIR)

    wavs = sorted([p for p in RANDOM_DIR.glob("*.wav") if p.is_file()])
    print(f"[random] Found {len(wavs)} raw files")

    total = 0
    cfg = {
        "window_s": WINDOW_S,
        "pre_s": PRE_S,
        "min_gap_s": MIN_GAP_S,
        "frame_ms": FRAME_MS,
        "hop_ms": HOP_MS,
        "thresh_mult": THRESH_MULT,
        "max_slices": MAX_SLICES_PER_FILE,
    }

    for wav in wavs:
        n = slice_keyword_file(wav, RANDOM_SPLICED_DIR, cfg)
        print(f"[random] {wav.name}: {n} slices")
        total += n

    print(f"[random] Total slices: {total}")

# =========================================================
# STEP 3: SPLIT BACKGROUND NOISE TO 1-SECOND FILES
# =========================================================
def split_noise_folder(noise_label: str):
    src_dir = UNKNOWN_DIR / noise_label
    out_dir = src_dir / "spliced"
    ensure_clean_dir(out_dir)

    wavs = sorted([p for p in src_dir.glob("*.wav") if p.is_file()])
    print(f"[{noise_label}] Found {len(wavs)} raw noise files")

    total = 0
    for wav_path in wavs:
        fs, data = wavfile.read(wav_path)
        data = to_mono(data).astype(np.int16)

        samples_per_chunk = int(fs * SAMPLE_TIME)
        n_chunks = len(data) // samples_per_chunk

        for i in range(n_chunks):
            start = i * samples_per_chunk
            end = start + samples_per_chunk
            chunk = data[start:end]

            out_name = f"{wav_path.stem}_u{i+1:02d}.wav"
            wavfile.write(out_dir / out_name, fs, chunk)
            total += 1

        print(f"[{noise_label}] {wav_path.name}: {n_chunks} chunks")

    print(f"[{noise_label}] Total chunks: {total}")

# =========================================================
# STEP 4: BUILD prepared_words
# =========================================================
def build_prepared_words():
    ensure_clean_dir(PREPARED_WORDS_DIR)

    for label in TARGET_LABELS:
        (PREPARED_WORDS_DIR / label).mkdir(parents=True, exist_ok=True)

    wavs = sorted([p for p in KEYWORDS_SPLICED_DIR.glob("*.wav") if p.is_file()])
    moved = 0

    for wav in wavs:
        m = SPLICED_FNAME_RE.match(wav.name)
        if not m:
            print(f"[prepared_words] Could not parse filename: {wav.name}")
            continue

        label = m.group("label")
        if label in TARGET_LABELS:
            shutil.copy2(wav, PREPARED_WORDS_DIR / label / wav.name)
            moved += 1
        else:
            print(f"[prepared_words] Ignored non-target label: {wav.name}")

    print(f"[prepared_words] Copied {moved} files")

    for label in TARGET_LABELS:
        count = len(list((PREPARED_WORDS_DIR / label).glob("*.wav")))
        print(f"[prepared_words] {label}: {count} files")

UNKNOWN_BALANCED_DIR = ROOT / "unknown_balanced"

def build_unknown_balanced(total_unknown_samples=300, random_ratio=0.8):
    ensure_clean_dir(UNKNOWN_BALANCED_DIR)

    n_random = int(total_unknown_samples * random_ratio)
    n_stilhed = total_unknown_samples - n_random

    random_files = sorted(RANDOM_SPLICED_DIR.glob("*.wav"))
    stilhed_dir = UNKNOWN_DIR / "stilhed" / "spliced"
    stilhed_files = sorted(stilhed_dir.glob("*.wav"))

    if len(random_files) < n_random:
        raise ValueError(f"Not enough random samples: need {n_random}, found {len(random_files)}")
    if len(stilhed_files) < n_stilhed:
        raise ValueError(f"Not enough stilhed samples: need {n_stilhed}, found {len(stilhed_files)}")

    selected_random = random.sample(random_files, n_random)
    selected_stilhed = random.sample(stilhed_files, n_stilhed)

    for wav in selected_random + selected_stilhed:
        shutil.copy2(wav, UNKNOWN_BALANCED_DIR / wav.name)

    print(f"[unknown_balanced] random:  {len(selected_random)}")
    print(f"[unknown_balanced] stilhed: {len(selected_stilhed)}")
    print(f"[unknown_balanced] total:   {len(selected_random) + len(selected_stilhed)}")
    
# =========================================================
# MAIN
# =========================================================
def main():
    reset_output_dirs()
    splice_keywords()
    splice_random_words()

    for unknown_label in UNKNOWN_LABELS:
        if unknown_label != "random":
            split_noise_folder(unknown_label)

    build_prepared_words()
    build_unknown_balanced()
    print("\nPipeline complete.")

if __name__ == "__main__":
    main()

[reset] Deleting: C:\Users\chris\OneDrive - Aarhus universitet\Skrivebord\Valgfag\F26\TML\Projekt\Dataopsamling\AutomaticCuration\keywords\spliced
[reset] Deleting: C:\Users\chris\OneDrive - Aarhus universitet\Skrivebord\Valgfag\F26\TML\Projekt\Dataopsamling\AutomaticCuration\unknown\random\spliced
[reset] Deleting: C:\Users\chris\OneDrive - Aarhus universitet\Skrivebord\Valgfag\F26\TML\Projekt\Dataopsamling\AutomaticCuration\prepared_words
[reset] Deleting: C:\Users\chris\OneDrive - Aarhus universitet\Skrivebord\Valgfag\F26\TML\Projekt\Dataopsamling\AutomaticCuration\background_noise
[reset] Deleting: C:\Users\chris\OneDrive - Aarhus universitet\Skrivebord\Valgfag\F26\TML\Projekt\Dataopsamling\AutomaticCuration\keywords_augmented
[reset] Deleting: C:\Users\chris\OneDrive - Aarhus universitet\Skrivebord\Valgfag\F26\TML\Projekt\Dataopsamling\AutomaticCuration\unknown\nygaard\spliced
[reset] Deleting: C:\Users\chris\OneDrive - Aarhus universitet\Skrivebord\Valgfag\F26\TML\Projekt\Dataops